In [ ]:
from PIL import Image
import torch
import numpy as np
import matplotlib.pylab as plt

import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
#from monai.networks.nets import UNet
#from monai.transforms import Compose, ToTensor
from monai.inferers import SlidingWindowInferer

from utils import model_fn

from image_processing import (
    normalize,
    get_glasbey_cmap,
    postprocessing_sdt,
    last_layer_fn,
)

In [ ]:
# path to example image
#path = r"D:\Institute of Genetics and Cancer\EM Alicante Datasets\L51L1_4-ds3_test.tif"
#path_labels = r"D:\Institute of Genetics and Cancer\EM Alicante Datasets\L51L1_4-ds3_test.tif"
#path_semantic = r'D:\Institute of Genetics and Cancer\EM Alicante Datasets\L51L1_4-ds3_test.tif'

path = r"sem_test/SEM Imaging_T1-A+B_q1_x01_y01_s0436_5.tif"
path_labels = r"sem_test/SEM Imaging_T1-A+B_q1_x01_y01_s0436_5.tif"
path_semantic = r'sem_test/SEM Imaging_T1-A+B_q1_x01_y01_s0436_5.tif'

In [ ]:
# Load image and normalize
image = np.array(Image.open(path)).astype(np.float32)
image = normalize(image)

# Load ground truth
labels = np.array(Image.open(path_labels)).astype(np.float32)
semantic = np.array(Image.open(path_semantic)).astype(np.float32)

In [ ]:
# Create the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model_fn(device)

# Load state_dict
model.load_state_dict(torch.load(r"last_epoch_model.pth"))

# inferer
inferer = SlidingWindowInferer(roi_size=(1024, 1024), sw_batch_size=1, overlap=0.25, progress=True)

In [ ]:
# Transform
data = torch.tensor(image).to(device)[(None,)*2] # unsqueeze(0) twice
print(data.shape)

In [ ]:
# Set model to eval
model.eval()
with torch.no_grad():
    pred = inferer(inputs=data, network=model)

In [ ]:
plt.imshow(pred[:, 4, :, :].squeeze().detach().cpu().numpy())

In [ ]:
# tensor to numpy
output = pred[:, 4, :, :].squeeze().detach().cpu().numpy()
postprocessed = postprocessing_sdt(pred[:, 4, :, :])

# show result
glasbey = get_glasbey_cmap()
fig, ax = plt.subplots(1,3, figsize=(14,4))
fig.suptitle("Show results")
ax[0].imshow(image, cmap='gray')
ax[0].set_title("Image")
ax[1].imshow(output, cmap='plasma')
ax[1].set_title("Prediction")
ax[2].imshow(postprocessed, cmap=glasbey, interpolation='nearest')
ax[2].set_title("Segmentation")
plt.tight_layout
plt.show()

In [ ]:
# tensor to numpy
output = pred[:, 4, :, :].squeeze().detach().cpu().numpy()
postprocessed = postprocessing_sdt(pred[:, 4, :, :])
pred_semantic = last_layer_fn(pred[:, 0:4, :, :]).squeeze().detach().cpu().numpy()

# show result
glasbey = get_glasbey_cmap()
fig, ax = plt.subplots(2,3, figsize=(16,10))
fig.suptitle("Show results")
ax[0,0].imshow(image, cmap='gray')
ax[0,0].set_title("Image")
ax[0,1].imshow(semantic, cmap='viridis', interpolation='nearest')
ax[0,1].set_title("Target Semantic")
ax[0,2].imshow(labels, cmap=glasbey, interpolation='nearest')
ax[0,2].set_title("Target Instance")
ax[1,0].imshow(output, cmap='plasma')
ax[1,0].set_title("Prediction Distance Transform")
ax[1,1].imshow(pred_semantic, cmap='viridis', interpolation='nearest')
ax[1,1].set_title("Prediction Semantic")
ax[1,2].imshow(postprocessed, cmap=glasbey, interpolation='nearest')
ax[1,2].set_title("Prediction Instance")
plt.tight_layout
plt.show()

In [ ]:
fig.savefig(r'figures\test_output.png', bbox_inches='tight', dpi=300)

In [ ]:
# Save intermediate and final results as tif files

# input image (normalized)
input = Image.fromarray(image, 'F') # 32-bit
input.save('figures\input.tif')

# semantic, distance transform and instance predictions
image_pred_semantic = Image.fromarray(pred_semantic.astype(np.uint8), 'L') # 8-bit
image_pred_semantic.save('figures\pred_semantic.tif')
image_pred_distrans = Image.fromarray(output, 'F') # 32-bit
image_pred_distrans.save('figures\pred_distrans.tif')
image_pred_instance = Image.fromarray(postprocessed, 'I') # 16-bit
image_pred_instance.save('figures\pred_instance.tif')

# watershed seeds
def postprocessing_sdt_seeds(distancemap: torch.Tensor):
    from skimage.measure import label
    from skimage.morphology import remove_small_objects

    distancemap = distancemap.squeeze().detach().cpu().numpy()
    mask_fibre = distancemap >= 0
    distancemap[distancemap < 0] = 0
    distancemap[distancemap > 1] = 1
    distancemap = normalize(distancemap)
    mask_inreg = np.logical_and(distancemap >= 0.7, mask_fibre)
    mask_inreg = label(mask_inreg)
    mask_inreg = remove_small_objects(mask_inreg, min_size=100, connectivity=1)
    mask_inreg = label(mask_inreg)
    
    return mask_inreg

seeds = postprocessing_sdt_seeds(pred[:, 4, :, :])
image_seeds = Image.fromarray(seeds.astype(np.uint16), 'I;16') # 16-bit
image_seeds.save('figures\seeds.tif')

# probability map (semantic) independent channels
probability_map = pred[:, 0:4, :, :].squeeze().detach().cpu().numpy()
pmap1 = Image.fromarray(probability_map[0, :, :], 'F') # 32-bit
pmap1.save('figures\probmap_ch1_bg.tif')
pmap2 = Image.fromarray(probability_map[1, :, :], 'F') # 32-bit
pmap2.save('figures\probmap_ch2_myelin.tif')
pmap3 = Image.fromarray(probability_map[2, :, :], 'F') # 32-bit
pmap3.save('figures\probmap_ch3_innertongue.tif')
pmap4 = Image.fromarray(probability_map[3, :, :], 'F') # 32-bit
pmap4.save('figures\probmap_ch4_axon.tif')